In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

DATA_DIR = os.path.join(r"D:\Upskill\Mini_Projects\intelligent-predictive-maintenance-system\CMAPSS_Data")

In [34]:
train_data = pd.read_csv(os.path.join(DATA_DIR, "train_FD001.txt"), sep=" ", header=None)
train_data = train_data.dropna(axis=1)

#Get the column names from the dataset documentation
column_names = ["engine_id", "cycle"] + [f"operational_setting_{i}" for i in range(1, 4)] + [f"sensor_{i}" for i in range(1, 22)]

#Assign column names to the DataFrame
train_data.columns = column_names

In [35]:
# Finding RUL (Remaining Useful Life) using the given data
max_cycles = train_data.groupby('engine_id')["cycle"].max() #Give max cycles of each engine as a pandas Series
print(max_cycles)
print('---------------------------------------------')

# Adding an RUL column that tells how close an engine is to failure. Creating ground truth for the training
train_data['RUL'] = train_data.apply(lambda row:max_cycles[row['engine_id']] - row['cycle'], axis = 1)
print(train_data[['engine_id','cycle', 'RUL']])
print('---------------------------------------------')

# Adding a boolean column to tell if the engine is close to failure or not based on a threshold
threshold = 30
train_data['failure_risk'] = (train_data['RUL'] <= 30).astype(int) # Convert to int as we need numeric data to train model
print(train_data.loc[train_data['engine_id'] == 1, ['engine_id','cycle', 'RUL', 'failure_risk']]) #.loc[] takes input as [row, column]. filter by row first then by column

engine_id
1      192
2      287
3      179
4      189
5      269
      ... 
96     336
97     202
98     156
99     185
100    200
Name: cycle, Length: 100, dtype: int64
---------------------------------------------
       engine_id  cycle    RUL
0              1      1  191.0
1              1      2  190.0
2              1      3  189.0
3              1      4  188.0
4              1      5  187.0
...          ...    ...    ...
20626        100    196    4.0
20627        100    197    3.0
20628        100    198    2.0
20629        100    199    1.0
20630        100    200    0.0

[20631 rows x 3 columns]
---------------------------------------------
     engine_id  cycle    RUL  failure_risk
0            1      1  191.0             0
1            1      2  190.0             0
2            1      3  189.0             0
3            1      4  188.0             0
4            1      5  187.0             0
..         ...    ...    ...           ...
187          1    188    4.0           

In [36]:
# Adding RUL for test dataset in the same way

test_data = pd.read_csv(os.path.join(DATA_DIR, "test_FD001.txt"), sep=" ", header=None)
RUL_data = pd.read_csv(os.path.join(DATA_DIR, "RUL_FD001.txt"), header=None)

test_data.dropna(axis=1, inplace=True)
RUL_data.dropna(axis=1, inplace=True)

test_data.columns = column_names

max_cycles = test_data.groupby('engine_id')['cycle'].max()
max_cycles = max_cycles + RUL_data[0].values

test_data['RUL'] = test_data.apply(lambda row:max_cycles[row['engine_id']] - row['cycle'], axis = 1)

threshold = 30
test_data['failure_risk'] = (test_data['RUL'] <= 30).astype(int) # Convert to int as we need numeric data to train model

## Setting up PySpark

In [37]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, lag, stddev, col
import os, sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

#Creating spark session
spark = (
    SparkSession.builder
    .master("local[1]")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "true")
    .config("spark.executor.instances", "1")
    .getOrCreate()
)

In [38]:
train_spark = spark.createDataFrame(train_data)
test_spark = spark.createDataFrame(test_data)

In [39]:
# Define a window over which all operations will be performed
window = Window.partitionBy("engine_id").orderBy("cycle") # For each engine, compute features in time order

# Define a rolling window
rolling_window = Window.partitionBy("engine_id") \
                       .orderBy("cycle") \
                       .rowsBetween(-4, 0) #current row + previous 4 rows

## Feature Engineering


In [40]:
#Computing lag1, lag2, mean, and std dev for each sensor in training dataframe
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
for sensor in sensor_cols:

    train_spark = train_spark.withColumn(
        sensor,
        col(sensor).cast("double")
    )

    # Lag 1 (cycle 10 → value from cycle 9)
    train_spark = train_spark.withColumn(
        f"{sensor}_lag1",
        lag(sensor, 1).over(window)
    )

    # Lag 2 (cycle 10 → value from cycle 9)
    train_spark = train_spark.withColumn(
        f"{sensor}_lag2",
        lag(sensor, 2).over(window)
    )

    # Rolling Mean
    train_spark = train_spark.withColumn(
        f"{sensor}_mean5",
        avg(sensor).over(rolling_window)
    )

    # Rolling Std Dev
    train_spark = train_spark.withColumn(
        f"{sensor}_std5",
        stddev(sensor).over(rolling_window)
    )

train_spark = train_spark.dropna()

In [41]:
#Computing lag1, lag2, mean, and std dev for each sensor in testing dataframe

for sensor in sensor_cols:

    test_spark = test_spark.withColumn(
        sensor,
        col(sensor).cast("double")
    )

    # Lag 1 (cycle 10 → value from cycle 9)
    test_spark = test_spark.withColumn(
        f"{sensor}_lag1",
        lag(sensor, 1).over(window)
    )

    # Lag 2 (cycle 10 → value from cycle 9)
    test_spark = test_spark.withColumn(
        f"{sensor}_lag2",
        lag(sensor, 2).over(window)
    )

    # Rolling Mean
    test_spark = test_spark.withColumn(
        f"{sensor}_mean5",
        avg(sensor).over(rolling_window)
    )

    # Rolling Std Dev
    test_spark = test_spark.withColumn(
        f"{sensor}_std5",
        stddev(sensor).over(rolling_window)
    )

test_spark = test_spark.dropna()

In [42]:
print("Training shape: ", train_spark.count(), len(train_spark.columns))
print("Testing shape: ", test_spark.count(), len(test_spark.columns))

Training shape:  20431 112
Testing shape:  12896 112


In [43]:
# Converting PySpark dataframe back to pandas dataframe to train model
train_df = train_spark.toPandas()
test_df = test_spark.toPandas()

train_df.to_csv(os.path.join(DATA_DIR, "processed", "train_features.csv"), index=False)
test_df.to_csv(os.path.join(DATA_DIR, "processed", "test_features.csv"), index=False)

In [26]:
# Separating input from output
y_train = train_df['failure_risk']
y_test = test_df['failure_risk']

drop_columns = ['engine_id', 'cycle', 'RUL', 'failure_risk']
x_train = train_df.drop(columns=drop_columns)
x_test = test_df.drop(columns=drop_columns)

In [27]:
print("x_train: ", x_train.shape)
print("y_train: ", y_train.shape)
print("x_test: ", x_test.shape)
print("y_test: ", y_test.shape)

x_train:  (20431, 108)
y_train:  (20431,)
x_test:  (12896, 108)
y_test:  (12896,)


In [32]:
import json

for i in range(-1, -10, -1):
    row = x_test.iloc[i].to_dict()
    print(json.dumps(row))

{"operational_setting_1": 0.0013, "operational_setting_2": 0.0003, "operational_setting_3": 100.0, "sensor_1": 518.67, "sensor_2": 642.95, "sensor_3": 1601.62, "sensor_4": 1424.99, "sensor_5": 14.62, "sensor_6": 21.61, "sensor_7": 552.48, "sensor_8": 2388.06, "sensor_9": 9155.03, "sensor_10": 1.3, "sensor_11": 47.8, "sensor_12": 521.07, "sensor_13": 2388.05, "sensor_14": 8214.64, "sensor_15": 8.4903, "sensor_16": 0.03, "sensor_17": 396.0, "sensor_18": 2388.0, "sensor_19": 100.0, "sensor_20": 38.7, "sensor_21": 23.1855, "sensor_1_lag1": 518.67, "sensor_1_lag2": 518.67, "sensor_1_mean5": 518.67, "sensor_1_std5": 0.0, "sensor_2_lag1": 643.26, "sensor_2_lag2": 643.44, "sensor_2_mean5": 643.222, "sensor_2_std5": 0.17555625878905015, "sensor_3_lag1": 1594.99, "sensor_3_lag2": 1593.15, "sensor_3_mean5": 1596.98, "sensor_3_std5": 3.461343669732828, "sensor_4_lag1": 1419.36, "sensor_4_lag2": 1406.82, "sensor_4_mean5": 1417.802, "sensor_4_std5": 7.014140717151307, "sensor_5_lag1": 14.62, "sensor

## Setting Up MLFlow

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Predictive_Maintenance_FD001")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778401818389, experiment_id='1', last_update_time=1778401818389, lifecycle_stage='active', name='Predictive_Maintenance_FD001', tags={}, trace_location=None, workspace='default'>

# Model training and Experimentation

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


In [ ]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

### Training Logistic Regression model

In [ ]:
with mlflow.start_run(run_name="LogisticRegression"):
    # Model
    model = LogisticRegression(
        max_iter=1000
    )

    # Train
    model.fit(x_train_scaled, y_train)

    # Predict
    y_pred = model.predict(x_test_scaled)
    y_prob = model.predict_proba(x_test_scaled)[:,1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)    
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    #Confusion Matrix
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    plt.savefig("confusion_matrix.png")
    plt.close(fig)

    # Log params
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "none")
    mlflow.log_param("max_iter", 1000)

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)

    # Log model and artifacts
    mlflow.sklearn.log_model(model, "logistic_regression_model")
    mlflow.log_artifact("confusion_matrix.png")

2026/05/10 14:30:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:30:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/1/runs/f74112c5801e4f259b68b3a2127b9af8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Training Logistic Regression model with balanced class weight

In [ ]:
with mlflow.start_run(run_name="LogisticRegression_Balanced"):
    # Model
    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000
    )

    # Train
    model.fit(x_train_scaled, y_train)

    # Predict
    y_pred = model.predict(x_test_scaled)
    y_prob = model.predict_proba(x_test_scaled)[:,1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)    
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    #Confusion Matrix
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    plt.savefig("confusion_matrix.png")
    plt.close(fig)

    # Log params
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("max_iter", 1000)

    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)

    # Log model and artifacts
    mlflow.sklearn.log_model(model, "logistic_regression_balanced_model")
    mlflow.log_artifact("confusion_matrix.png")

2026/05/10 14:30:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:30:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression_Balanced at: http://127.0.0.1:5000/#/experiments/1/runs/2171d6c299c34633b4a6fcccfa0a351a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Training Random Forest model and checking performance with different threshold values

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Model
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced",
    random_state=42
)

# Random forest works with unscaled data and it is robust to noisy sensors
rf.fit(x_train, y_train)

y_prob = rf.predict_proba(x_test)[:,1]

#Creating feature importance csv
importance = pd.Series(
    rf.feature_importances_,
    index=x_train.columns
).sort_values(ascending=False)
importance_df = importance.reset_index()
importance_df.columns = ["feature", "importance"]
importance_df.to_csv("feature_importance.csv", index=False)

for threshold in [0.3, 0.4, 0.5, 0.6]:
    y_pred = (y_prob >= threshold).astype(int) #If probability > threshold, predict as failure.
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    #Confusion Matrix
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    plt.savefig(f"confusion_matrix.png")
    plt.close(fig)

    with mlflow.start_run(run_name=f"RandomForest_threshold_{threshold}"):
        # Log params
        mlflow.log_param("model_type", "RandomForest")
        mlflow.log_param("class_weight", "balanced")
        mlflow.log_param("n_estimators", 200)        
        mlflow.log_param("max_depth", "none")
        mlflow.log_param("random_state", 42)
        mlflow.log_param("threshold", threshold)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("pr_auc", pr_auc)

        # Log model and artifacts
        mlflow.log_artifact("confusion_matrix.png")
        mlflow.log_artifact("feature_importance.csv")
        mlflow.sklearn.log_model(rf, "random_forest_model")

2026/05/10 14:30:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:30:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_threshold_0.3 at: http://127.0.0.1:5000/#/experiments/1/runs/a0eb60e7be04417fa652b08a12e372b9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/05/10 14:30:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:30:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_threshold_0.4 at: http://127.0.0.1:5000/#/experiments/1/runs/1b54b921e5a449888b31786c8348d120
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/05/10 14:30:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:30:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_threshold_0.5 at: http://127.0.0.1:5000/#/experiments/1/runs/c03924d9f40b4d418c076d6f9425f091
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/05/10 14:30:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:30:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest_threshold_0.6 at: http://127.0.0.1:5000/#/experiments/1/runs/29e0a99369634c04a02dbd4cda6907b6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


### Training XGBoost Model and checking performance with different threshold values

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

# Model
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1
)

# XGBoost handles nonlinear feature interactions and performs well on tabular sensor data
xgb.fit(x_train, y_train)

y_prob = xgb.predict_proba(x_test)[:,1]

#Creating feature importance csv
importance = pd.Series(
    xgb.feature_importances_,
    index=x_train.columns,
).sort_values(ascending=False)
importance_df = importance.reset_index()
importance_df.columns = ["feature", "importance"]
importance_df.to_csv("feature_importance.csv", index=False)

for threshold in [0.3, 0.4, 0.5, 0.6]:
    y_pred = (y_prob >= threshold).astype(int) #If probability > threshold, predict as failure.
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)

    #Confusion Matrix
    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    plt.savefig(f"confusion_matrix.png")
    plt.close(fig)

    with mlflow.start_run(run_name=f"xgboost_threshold_{threshold}"):
        # Log params
        mlflow.log_param("model_type", "XGBoost")
        mlflow.log_param("scale_pos_weight", scale_pos_weight)
        mlflow.log_param("n_estimators", 200)        
        mlflow.log_param("max_depth", 6)
        mlflow.log_param("learning_rate", 0.05)
        mlflow.log_param("subsample", 0.8)
        mlflow.log_param("colsample_bytree", 0.8)
        mlflow.log_param("random_state", 42)
        mlflow.log_param("eval_metric", "logloss")
        mlflow.log_param("threshold", threshold)

        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("pr_auc", pr_auc)

        # Log model and artifacts
        mlflow.log_artifact("confusion_matrix.png")
        mlflow.log_artifact("feature_importance.csv")
        mlflow.sklearn.log_model(xgb, "xgboost_model")

2026/05/10 14:30:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:30:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_threshold_0.3 at: http://127.0.0.1:5000/#/experiments/1/runs/b8062d0813c54d2785df170277cc37ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/05/10 14:31:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:31:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_threshold_0.4 at: http://127.0.0.1:5000/#/experiments/1/runs/b6799e14a4d649028f0b02cd4d5d29cb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/05/10 14:31:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:31:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_threshold_0.5 at: http://127.0.0.1:5000/#/experiments/1/runs/ce8b1b669f98479785be57a71cd8f078
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/05/10 14:31:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 14:31:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_threshold_0.6 at: http://127.0.0.1:5000/#/experiments/1/runs/2e0164abaa344d6f9c88f989b1caee0e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
